In [2]:
import numpy as np
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

ModuleNotFoundError: No module named 'tensorflow'

In [3]:
# Create directories for data storage
data_dir = "memory_data"
classes = ["memory_leak", "memory_spikes", "no_issue"]

for cls in classes:
    os.makedirs(os.path.join(data_dir, cls), exist_ok=True)

# Function to generate synthetic memory usage graphs
def generate_memory_chart(label, save_path):
    num_points = 1000
    x = np.linspace(0, num_points, num_points)
    y = np.zeros_like(x)

    trend = np.random.uniform(-0.03, 0.03, num_points).cumsum()
    y = np.random.randint(10,50) + trend * 10 + np.random.normal(0, np.random.randint(2,7), num_points)

    if label == "memory_leak":
        # More controlled gradual increase with irregular pattern

        noise = np.linspace(np.random.uniform(0, 15), np.random.uniform(40, 60), num_points)
        y += noise



    elif label == "memory_spikes":
        num_spikes = np.random.randint(3, 10)  # Random number of spikes
        for _ in range(num_spikes):
            start = np.random.randint(100, 900)
            duration = np.random.randint(5, 50)  # Random duration of spike
            y[start:start+duration] += np.random.randint(30, 80)  # Random spike intensity


    plt.figure(figsize=(6, 3))
    plt.plot(x, y, color='blue')
    plt.xlabel("Time")
    plt.ylabel("Memory Usage")
    plt.ylim(0, 120)
    plt.axis('off')  # Hide axes for better classification
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()

In [4]:
# Generate and save images
num_samples_for_each_class = 1500
for cls in classes:
    for i in range(num_samples_for_each_class):
        save_path = os.path.join(data_dir, cls, f"{cls}_{i}.png")
        generate_memory_chart(cls, save_path)

In [5]:
# Data augmentation with stretching, scaling, and chart-related transformations
datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.2,
    width_shift_range=0.2,  # Horizontal shifting
    height_shift_range=0.2,  # Vertical shifting
    zoom_range=[0.8, 1.2],  # Scaling
    shear_range=0.2,  # Skewing effect
    rotation_range=10,  # Small rotations to simulate chart variations
    horizontal_flip=False  # Flipping is not relevant for time-series data
)
train_generator = datagen.flow_from_directory(
    data_dir, target_size=(64, 64), batch_size=32, class_mode='categorical', subset='training')

val_generator = datagen.flow_from_directory(
    data_dir, target_size=(64, 64), batch_size=32, class_mode='categorical', subset='validation')

Found 3600 images belonging to 3 classes.
Found 900 images belonging to 3 classes.


In [14]:
# CNN Model
model = Sequential([
    Conv2D(8, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Conv2D(16, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Conv2D(32, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.3),
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')  # 3 classes
])

# Compile the model with a lower learning rate for better convergence
model.compile(optimizer=Adam(learning_rate=0.0005), loss='categorical_crossentropy', metrics=['accuracy'])

In [15]:
# Train the model
model.fit(train_generator, validation_data=val_generator, epochs=10)

Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 34s 257ms/step - accuracy: 0.6965 - loss: 0.7629 - val_accuracy: 0.3333 - val_loss: 1.4353
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 29s 259ms/step - accuracy: 0.9250 - loss: 0.1994 - val_accuracy: 0.3333 - val_loss: 1.7050
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 28s 244ms/step - accuracy: 0.9307 - loss: 0.1856 - val_accuracy: 0.8000 - val_loss: 0.4729
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 28s 249ms/step - accuracy: 0.9444 - loss: 0.1425 - val_accuracy: 0.9133 - val_loss: 0.2245
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 28s 247ms/step - accuracy: 0.9515 - loss: 0.1232 - val_accuracy: 0.9578 - val_loss: 0.1209
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 28s 250ms/step - accuracy: 0.9454 - loss: 0.1512 - val_accuracy: 0.9311 - val_loss: 0.1658
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 29s 260ms/step - accuracy: 0.9634 - loss: 0.1072 - val_accuracy: 0.9389 - val_loss: 0.1795
Epoch 8/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 40s 247ms/step - accuracy: 0.9576 - loss: 0

In [16]:
# Save the model
model.save("memory_classification_model.h5")

In [1]:
import shutil
shutil.rmtree("memory_data")


In [28]:
from tensorflow.keras.models import load_model
loaded_model = load_model("memory_classification_model.h5")


In [29]:
import cv2

def preprocess_image(image_path):
    img = cv2.imread(image_path)  # Load the image
    img = cv2.resize(img, (64, 64))  # Resize to match the input shape
    img = img / 255.0  # Normalize pixel values
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    return img


image_path = "memory_data/memory_spikes/memory_spikes_100.png"
processed_image = preprocess_image(image_path)

prediction = loaded_model.predict(processed_image)
class_index = np.argmax(prediction)  # Get the predicted class index

# Map index to class label
class_labels = ["memory_leak", "memory_spikes", "no_issue"]
predicted_label = class_labels[class_index]

print(f"Predicted Class: {predicted_label}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 214ms/step
Predicted Class: memory_spikes


In [30]:
for i, label in enumerate(class_labels):
    print(f"{label}: {prediction[0][i]*100:.2f}% confidence")


memory_leak: 0.00% confidence
memory_spikes: 100.00% confidence
no_issue: 0.00% confidence
